In [ ]:
from Crypto.PublicKey import RSA
from Crypto.Cipher import AES, PKCS1_OAEP
import socket

HOST = "127.0.0.1"
PORT = 40003

private_key = RSA.import_key(open("private.pem").read())

if __name__ == "__main__":
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    server.bind((HOST, PORT))
    server.listen(0)

    while True:
        connection, client_address = server.accept()

        
        enc_session_key = connection.recv(private_key.size_in_bytes())
        nonce = connection.recv(16)
        tag = connection.recv(16)
        ciphertext = connection.recv(1000)

        cipher_rsa = PKCS1_OAEP.new(private_key)
        session_key = cipher_rsa.decrypt(enc_session_key)

        # Decrypt the data with the AES session key
        cipher_aes = AES.new(session_key, AES.MODE_EAX, nonce)
        data = cipher_aes.decrypt_and_verify(ciphertext, tag)
        print(data.decode("utf-8"))

Jov, odkril sem skrivni nacrt za Slovenijo 2.


### Creating a Socket

```python
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
```

- `socket.socket(socket.AF_INET, socket.SOCK_STREAM)` creates a new socket object.
- `socket.AF_INET` specifies that the socket will use the IPv4 address family.
- `socket.SOCK_STREAM` specifies that the socket type will be TCP (a stream socket), which provides reliable, ordered, and error-checked delivery of data.

### Setting Socket Options

```python
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
```

- `server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)` sets the socket option `SO_REUSEADDR`.
- This allows the socket to be bound to an address that is already in use, which is useful during development to quickly restart the server without waiting for the OS to release the port.

### Binding the Socket

```python
server.bind((HOST, PORT))
```

- `server.bind((HOST, PORT))` binds the socket to the specified host and port.
- `HOST` is the IP address (e.g., `'127.0.0.1'`), and `PORT` is the port number (e.g., `65432`).

### Listening for Connections

```python
server.listen(0)
```

- `server.listen(0)` enables the server to accept connections.
- The argument `0` specifies the maximum number of queued connections. `0` or leaving it empty typically means that the operating system will set a reasonable default value.

### Accepting Connections

```python
while True:
    connection, client_address = server.accept()
```

- This loop continuously waits for incoming client connections.
- `server.accept()` blocks and waits for an incoming connection.
- When a client connects, `server.accept()` returns a new socket object (`connection`) for the connection and the address of the client (`client_address`).

